In [5]:
import os
import json
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

class AsistenteMetroRAG:
    def __init__(self):
        load_dotenv()
        self.client = OpenAI(
            # Aquí van los NOMBRES de las variables que están en tu archivo .env
            base_url=os.environ.get("GITHUB_BASE_URL"), 
            api_key=os.environ.get("GITHUB_TOKEN")
        )
        # Base de conocimientos: Fragmentos de información técnica del Metro
        self.documentos = [
            "Tarifa PUNTA (07:00-08:59 y 18:00-19:59): Cuesta $840. Es el horario de la mañana y tarde con más gente.",
            "Tarifa VALLE (09:00-17:59 y 20:00-20:44): Cuesta $760. Horario intermedio.",
            "Tarifa BAJO (06:00-06:59 y 20:45-23:00): Cuesta $680. Horario temprano o muy tarde.",
            "Para ir a PUENTE ALTO: Debes tomar la Línea 4 (Azul) que llega hasta Plaza de Puente Alto.",
            "COMBINACIÓN: En la estación Tobalaba puedes cambiar entre Línea 1 y Línea 4 para ir a Puente Alto."
        ]
        
        self.embeddings_db = []
        self._preparar_rag()

    def _get_embedding(self, texto):
        """Genera la representación vectorial del texto (Concepto clave IL 1.3)"""
        response = self.client.embeddings.create(
            input=texto,
            model="text-embedding-3-small"
        )
        return response.data[0].embedding

    def _preparar_rag(self):
        """Indexación: Convierte toda la base de conocimientos en vectores"""
        print("Indexando base de conocimientos...")
        for doc in self.documentos:
            self.embeddings_db.append(self._get_embedding(doc))

    def _recuperar_contexto(self, consulta_usuario, top_k=4): # <--- Subimos a 4
        v_consulta = self._get_embedding(consulta_usuario)
        similitudes = [np.dot(v_consulta, v_db) for v_db in self.embeddings_db]
        
        # Obtener los mejores 4 fragmentos
        indices = np.argsort(similitudes)[-top_k:]
        return "\n".join([self.documentos[i] for i in indices])

    def consultar(self, pregunta):
        # 1. Recuperar contexto relevante mediante RAG
        contexto = self._recuperar_contexto(pregunta)
        
        # 2. Construir el prompt con el contexto inyectado
        prompt_sistema = f"""
        Eres el asistente oficial de Metro de Santiago. 
        Usa SOLO la siguiente información recuperada para responder:
        {contexto}

        Responde siempre en formato JSON con estas llaves:
        - 'metacognicion': Tu razonamiento lógico.
        - 'respuesta': La respuesta final al usuario.
        - 'fuentes_usadas': Qué fragmentos del contexto utilizaste.
        """

        response = self.client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt_sistema},
                {"role": "user", "content": pregunta}
            ],
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)

# === PRUEBA DEL SISTEMA ===
if __name__ == "__main__":
    asistente = AsistenteMetroRAG()
    resultado = asistente.consultar("¿Cuánto me sale el pasaje a las 8 am y cómo llego a Puente Alto?")
    print(json.dumps(resultado, indent=4, ensure_ascii=False))

Indexando base de conocimientos...
{
    "metacognicion": "El horario de las 8 am corresponde a la tarifa PUNTA, y para llegar a Puente Alto necesitas tomar la Línea 4 (Azul). Si estás en una estación de la Línea 1, puedes combinar en Tobalaba.",
    "respuesta": "El pasaje te costará $840 a las 8 am porque corresponde al horario PUNTA. Para llegar a Puente Alto, debes tomar la Línea 4 (Azul), que llega a Plaza de Puente Alto. Si estás en la Línea 1, debes combinar en la estación Tobalaba para cambiar a la Línea 4.",
    "fuentes_usadas": "Tarifa PUNTA (07:00-08:59 y 18:00-19:59): Cuesta $840. Es el horario de la mañana y tarde con más gente; COMBINACIÓN: En la estación Tobalaba puedes cambiar entre Línea 1 y Línea 4 para ir a Puente Alto; Para ir a PUENTE ALTO: Debes tomar la Línea 4 (Azul) que llega hasta Plaza de Puente Alto."
}
